# S0 — 02.06: Lasso Regression (L1)

**Question:** How can an L1 penalty produce exact zeros in a regression coefficient while the model is being fitted?

This is a small synthetic training calculation. It is not evidence about feature importance, performance, or generalisation.

## S1 — Objective and sparsity

With `y_hat = b + X @ w`, this lesson uses `J = MSE + lambda * sum(abs(w))`. The intercept `b` is not penalised. Unlike L2 shrinkage, the L1 penalty can make a coefficient exactly zero.

In [ ]:
import numpy as np
X = np.array([[0.4082482904638631, 0.5], [0.4082482904638631, -0.5], [0.4082482904638631, 0.0], [-0.4082482904638631, 0.0], [-0.4082482904638631, 0.5], [-0.4082482904638631, -0.5]], dtype=np.float64)
y = np.array([0.712372, 0.512372, 1.887117, 1.787117, -1.757117, -1.917117], dtype=np.float64)
LAMBDA_DENSE, LAMBDA_SPARSE = 0.05, 0.20
print(X.shape, y.shape, X.T @ X)

## S2 — Soft thresholding

`S(z, gamma) = sign(z) * max(abs(z) - gamma, 0)`. For example, `S(-3, 1) = -2`, while `S(0.5, 1) = 0`. For this mean-scale objective, the update threshold is `N * lambda / 2`. This is a definition and an update mechanism, not a full subdifferential derivation.

In [ ]:
def soft_threshold(z, gamma):
    return np.sign(z) * np.maximum(np.abs(z) - gamma, 0.0)

def lasso_coordinate_descent(X, y, lam, max_iter=1000, tol=1e-12):
    X, y = np.asarray(X, dtype=np.float64), np.asarray(y, dtype=np.float64)
    if lam < 0 or max_iter <= 0 or X.ndim != 2 or y.ndim != 1 or X.shape[0] != y.size:
        raise ValueError('invalid Lasso inputs')
    b, w = np.mean(y), np.zeros(X.shape[1], dtype=np.float64)
    for _ in range(max_iter):
        old = np.r_[b, w].copy()
        for j in range(X.shape[1]):
            r = y - b - X @ w + X[:, j] * w[j]
            w[j] = soft_threshold(X[:, j] @ r, X.shape[0] * lam / 2) / (X[:, j] @ X[:, j])
        b = np.mean(y - X @ w)
        if np.max(np.abs(np.r_[b, w] - old)) < tol:
            break
    return np.r_[b, w]

for lam in (LAMBDA_DENSE, LAMBDA_SPARSE):
    theta = lasso_coordinate_descent(X, y, lam)
    predictions = theta[0] + X @ theta[1:]
    residuals = y - predictions
    rss = np.sum(residuals**2)
    mse = rss / y.size
    penalty = lam * np.sum(np.abs(theta[1:]))
    print(lam, theta, rss, mse, penalty, mse + penalty)

## S3 — Two coordinate-descent cycles

Start from `b=0` and `w=[0, 0]`. For each feature, remove its current contribution, correlate the partial residual, soft-threshold with `N*lambda/2`, and divide by its squared norm. The intercept is refreshed after the feature updates. The following trace is deliberately bounded; convergence is checked by code rather than assumed after a fixed number of cycles.

In [ ]:
def trace_cycle(X, y, lam, theta):
    b, w = theta[0], theta[1:].copy()
    threshold = X.shape[0] * lam / 2
    for j in range(X.shape[1]):
        partial = y - b - X @ w + X[:, j] * w[j]
        rho = X[:, j] @ partial
        old_w = w[j]
        w[j] = soft_threshold(rho, threshold) / (X[:, j] @ X[:, j])
        print(f'j={j}: rho={rho:.6f}, threshold={threshold:.6f}, '
              f'w_old={old_w:.6f}, w_new={w[j]:.6f}')
    b = np.mean(y - X @ w)
    predictions = b + X @ w
    residuals = y - predictions
    rss = np.sum(residuals**2)
    objective = rss / y.size + lam * np.sum(np.abs(w))
    return np.r_[b, w], predictions, rss, objective

theta_trace = np.zeros(3, dtype=np.float64)
for cycle in (1, 2):
    theta_trace, predictions, rss, objective = trace_cycle(
        X, y, LAMBDA_SPARSE, theta_trace
    )
    print(f'cycle={cycle}, theta={theta_trace}, RSS={rss:.6f}, J={objective:.6f}')

## S4 — Scale bridge and interpretation

This lesson writes the loss as `MSE`. scikit-learn writes the loss as `RSS/(2N)`, so the independent comparison uses `alpha = lambda / 2`. A zero coefficient is a mechanical result of this fit; it is not proof that the feature is irrelevant or uncorrelated with the target.

## S5 — Accessible coefficient path

![Lasso coefficient paths](assets/lasso_paths.png)

**Text equivalent:** At `lambda=0.05`, `w1=1.890824220927` and `w2=0.03`; at `lambda=0.20`, `w1=1.440824220927` and `w2=0.0`. The dashed-square `w2` path reaches the zero line at the sparse teaching value while `w1` remains positive. This is a property of the synthetic training table, not evidence of feature irrelevance or generalisation.

## S6 — Practice and boundary

Explain why the intercept is excluded, distinguish soft from hard thresholding, and identify which coefficient is zero at `lambda = 0.20`. Larger lambda changes the training trade-off; it does not guarantee a better model. Feature scaling affects Lasso, and lambda is a hyperparameter rather than a fitted coefficient. Lasso is not always preferable to Ridge, and a zero coefficient is not proof of zero correlation or irrelevance.